## CONFIGURATION

In [174]:
CONFIG = {
    "EEG_data_path": "data/sliced_eeg/watching",
    "latent_data_path": "data/metadata/videos_latents.pt",
    "save_path": "checkpoints/seq2seq/best_seq2seq_model.pt",  # 更明确的最佳模型保存路径
    "result_path": "data/metadata/seq2seq_prediction.pt",

    "train_test": 22/25, 

    "video_latent_shape": (250, 12, 4, 36, 64),
    "eeg_t_window": 100,
    "eeg_t_step": 50,

    "d_model": 512,
    "eeg_channels": 62,
    
    "n_head": 4,
    "encoder_layers": 2,
    "decoder_layers": 4,
    "seed": 42,
    "dropout": 0.1,
    "batch_size": 128,
    "learning_rate": 0.001,
    "epochs": 100,

    "max_latent_seq_len": 6,
}

## Set seed

In [175]:
import torch
import numpy as np
import random
import os
import math
import json
from typing import Tuple, Any
from pathlib import Path

def set_seed(seed):
    """设置随机种子以确保实验的可重现性。"""
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(CONFIG["seed"])

## Define dataset

In [176]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, Dataset
from einops import rearrange, repeat
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

class EEGVideoDataset(Dataset):
    """Dataset for pairing EEG sequences with latent video sequences."""
    def __init__(self, eeg_data: torch.Tensor, video_data: torch.Tensor):
        self.eeg = eeg_data     # (60*220, 62, 7, 100) or (60*30, 62, 7, 100)
        self.video = video_data # (60*220, 12, 4, 36, 64) or (60*30, 12, 4, 36, 64)
        assert len(self.eeg) == len(self.video), "EEG and video data must have the same number of samples."

    def __len__(self) -> int:
        return len(self.eeg)

    def __getitem__(self, item: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.eeg[item], self.video[item]

def load_and_preprocess_data():
    """
    加载、预处理EEG和潜在向量数据，为训练和测试做准备。

    Returns:
        一个包含以下内容的元组:
        - DataLoader: 训练集的数据加载器
        - torch.Tensor: 预处理后的测试EEG数据
        - torch.Tensor: 预处理后的测试潜在向量数据
    """
    # --- Load Data ---
    eeg_data = []
    for file in os.listdir(CONFIG["EEG_data_path"]):
        eeg_data.append(np.load(os.path.join(CONFIG["EEG_data_path"], file)))
    eeg_data = np.stack(eeg_data) # (60, 5, 50, 62, 400)
    
    eeg_data = torch.from_numpy(eeg_data)
    windowed_eeg = eeg_data.unfold(dimension=-1, size=CONFIG["eeg_t_window"], step=CONFIG["eeg_t_step"]) # (60, 5, 50, 62, 400) -> (60, 5, 50, 62, 7, 100)
    windowed_eeg = rearrange(windowed_eeg, 'n g v c w t -> n (g v) c w t') # (60, 5*50, 62, 7, 100)
    n, v, c, w, t = windowed_eeg.shape # (60, 5*50, 62, 7, 100)
    
    # --- Split Train/Test and Reshape ---
    train_eeg = windowed_eeg[:, :int(windowed_eeg.shape[1]*CONFIG["train_test"]), ...] # (60, 220, 62, 7, 100)
    test_eeg = windowed_eeg[:, int(windowed_eeg.shape[1]*CONFIG["train_test"]): , ...] # (60, 30, 62, 7, 100)
    train_eeg = rearrange(train_eeg, 'n v c w t -> (n v c) (w t)') # (60*220*62, 7*100)
    test_eeg = rearrange(test_eeg, 'n v c w t -> (n v c) (w t)') # (60*30*62, 7*100)

    # Normalize EEG Data
    scaler = StandardScaler()
    train_eeg = scaler.fit_transform(train_eeg)
    test_eeg = scaler.transform(test_eeg)

    train_eeg = rearrange(train_eeg, '(nv c) (w t) -> nv w c t', w = w, c = c, t = t)
    test_eeg = rearrange(test_eeg, '(nv c) (w t) -> nv w c t', w = w, c = c, t = t)


    latent_data = torch.load(CONFIG["latent_data_path"]).cpu().numpy()
    train_latent_data = latent_data[:int(len(latent_data)*CONFIG["train_test"]), ...] # (220, 12, 4, 36, 64)
    test_latent_data = latent_data[int(len(latent_data)*CONFIG["train_test"]):, ...] # (30, 12, 4, 36, 64)
    train_latent_data = repeat(train_latent_data, 'v f c h w -> (n v) f c h w', n = n)
    test_latent_data = repeat(test_latent_data, 'v f c h w -> (n v) f c h w', n = n)

    print(f"Train EEG shape: {train_eeg.shape}") # (60*220, 7, 62, 100)
    print(f"Train Latent shape: {train_latent_data.shape}") # (60*220, 12, 4, 36, 64)
    print(f"Test EEG shape: {test_eeg.shape}") # (60*30, 7, 62, 100)
    print(f"Test Latent shape: {test_latent_data.shape}") # (60*30, 12, 4, 36, 64)

    # --- Create DataLoader ---
    train_dataset = EEGVideoDataset(train_eeg, train_latent_data)
    train_dataloader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)

    return train_dataloader, test_eeg, test_latent_data

## EEG embedding

In [177]:
import torch.nn as nn

class MyEEGNet_embedding(nn.Module):
    """
    EEGNet-based feature extractor to generate embeddings from raw EEG signals.

    This network follows the EEGNet architecture with temporal, depthwise, and
    separable convolutions to learn features from EEG data.

    Args:
        d_model (int): The dimensionality of the output embedding.
        C (int): The number of EEG channels.
        T (int): The number of time points in the EEG window.
        F1, D, F2 (int): Hyperparameters for the number of filters and depth
                         multiplier in the convolutional layers.
        cross_subject (bool): If True, uses a lower dropout rate suitable for
                              cross-subject generalization.
    """
    def __init__(self, d_model: int = 512, C: int = 62, T: int = 100, F1: int = 16, 
                 D: int = 4, F2: int = 16, cross_subject: bool = False):
        super().__init__()
        self.drop_out = 0.25 if cross_subject else 0.5
        
        # Temporal Convolution
        self.block_1 = nn.Sequential( # input (batch, 1, 62, 100)
            nn.ZeroPad2d((31, 32, 0, 0)),
            nn.Conv2d(1, F1, (1, 64), bias=False),
            nn.BatchNorm2d(F1)
        ) # output (batch, 16, 62, 100)
        
        # Depthwise Convolution
        self.block_2 = nn.Sequential(
            nn.Conv2d(F1, F1 * D, (C, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.Dropout(self.drop_out)# output (batch, 16*4, 1, 100)
        )
        
        # Separable Convolution
        self.block_3 = nn.Sequential(
            nn.ZeroPad2d((7, 8, 0, 0)),
            nn.Conv2d(F1 * D, F1 * D, (1, 16), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False), # (batch, 16, 1, 100)
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 2)),
            nn.Dropout(self.drop_out) # output (batch, 16, 1, 50)
        )
        
        # Final projection to d_model
        # F2 * (T // 2) = 16 * (100 // 2) = 800
        final_conv_output_size = F2 * (T // 2) 
        self.embedding = nn.Linear(final_conv_output_size, d_model) # output (batch, 512)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the EEGNet embedding.

        Args:
            x (torch.Tensor): Input EEG data of shape (batch, 1, C, T).

        Returns:
            torch.Tensor: Embedded EEG features of shape (batch, d_model).
        """
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.block_3(x)
        x = x.view(x.shape[0], -1)
        x = self.embedding(x)
        return x

## Position encoding

In [178]:
import torch
import torch.nn as nn
import math
# einops 在这个简化版本中不是必需的，但可以用来使维度操作更清晰
# from einops import rearrange 

class PositionalEncoding(nn.Module):
    """
    Injects positional information into the input embeddings.

    Args:
        d_model (int): The dimensionality of the embeddings. # (Batch, 7, d_model=512)
        dropout (float): The dropout rate.
        max_len (int): The maximum possible sequence length. # 100
    """
    def __init__(self, d_model: int, dropout: float, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # 创建一个 (max_len, d_model) 的位置编码矩阵 pe
        pe = torch.zeros(max_len, d_model)
        
        # 创建位置张量 (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # 创建维度除数项 (d_model,)
        div_term = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)

        # 使用广播机制计算 sin 和 cos
        # position (max_len, 1) * div_term (d_model/2,) -> angle (max_len, d_model/2)
        angle = position / div_term
        pe[:, 0::2] = torch.sin(angle)
        pe[:, 1::2] = torch.cos(angle)
        
        # 增加 batch 维度并注册为 buffer
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Input tensor of shape (batch, seq_len, d_model).

        Returns:
            torch.Tensor: Tensor with added positional information.
        """
        # 使用缓存的 pe，只截取需要的长度
        x = x + self.pe[:, :x.size(1)].requires_grad_(False)
        return self.dropout(x)

## Transformer to predict latent vectors

In [179]:
class LatentPredictionTransformer(nn.Module):
    """
    A Transformer model for predicting a sequence of latent video representations
    from a sequence of EEG window embeddings.

    Args:
        d_model (int): The main dimensionality of the model.
        n_head (int): The number of attention heads.
        encoder_layers (int): The number of layers in the Transformer encoder.
        decoder_layers (int): The number of layers in the Transformer decoder.
        dropout (float): The dropout rate.
        eeg_channels (int): Number of EEG channels.
        eeg_t_window (int): Time points per EEG window.
    """
    def __init__(self, d_model: int, n_head: int, encoder_layers: int, 
                 decoder_layers: int, dropout: float, eeg_channels: int, 
                 eeg_t_window: int):
        super().__init__()
        self.d_model = d_model
        
        # --- Embeddings ---
        # For EEG data (source sequence)
        self.eeg_embedding = MyEEGNet_embedding(d_model=d_model, C=eeg_channels, T=eeg_t_window)
        # For latent video frames (target sequence)
        self.latent_embedding = nn.Linear(4 * 36 * 64, d_model)
        
        # --- Positional Encoding ---
        self.positional_encoding = PositionalEncoding(d_model, dropout=dropout)
        
        # --- Transformer Core ---
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_head, batch_first=True, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=encoder_layers)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=n_head, batch_first=True, dropout=dropout)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=decoder_layers)
        
        # --- Output Predictors ---
        # Predicts a single class from the mean of encoder outputs
        self.txt_predictor = nn.Linear(d_model, 13) 
        # Predicts the next latent frame representation
        self.latent_predictor = nn.Linear(d_model, 4 * 36 * 64)

    def forward(self, src_eeg: torch.Tensor, tgt_latent: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass for training (uses teacher forcing).

        Args:
            src_eeg (torch.Tensor): Source EEG data. 
                Shape: (batch, seq_len_eeg, channels, time_points).
            tgt_latent (torch.Tensor): Target latent sequence (shifted right).
                Shape: (batch, seq_len_latent, 4, 36, 64).

        Returns:
            A tuple containing:
            - torch.Tensor: Text/class prediction. Shape: (batch, num_classes).
            - torch.Tensor: Predicted latent sequence. Shape: (batch, seq_len_latent, 4, 36, 64).
        """
        # 1. Process source (EEG) sequence
        b, seq_len_eeg, c, t = src_eeg.shape
        src_embedded = self.eeg_embedding(src_eeg.reshape(b * seq_len_eeg, 1, c, t))
        src_embedded = src_embedded.view(b, seq_len_eeg, self.d_model)
        src = self.positional_encoding(src_embedded)
        
        # 2. Process target (latent) sequence
        b, seq_len_latent, c, h, w = tgt_latent.shape
        tgt_flattened = tgt_latent.view(b, seq_len_latent, -1)
        tgt_embedded = self.latent_embedding(tgt_flattened)
        tgt = self.positional_encoding(tgt_embedded)

        # 3. Generate mask for the decoder
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len_latent).to(src.device)
        
        # 4. Run through Transformer
        encoder_output = self.transformer_encoder(src)
        decoder_output = self.transformer_decoder(tgt, encoder_output, tgt_mask=tgt_mask)
        
        # 5. Generate predictions
        txt_prediction = self.txt_predictor(torch.mean(encoder_output, dim=1))
        latent_prediction_flat = self.latent_predictor(decoder_output)
        latent_prediction = latent_prediction_flat.view(b, seq_len_latent, 4, 36, 64)
        
        return txt_prediction, latent_prediction

    def generate(self, src_eeg: torch.Tensor, max_len: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Auto-regressively generate a latent sequence for inference.

        Args:
            src_eeg (torch.Tensor): Source EEG data.
                Shape: (batch, seq_len_eeg, channels, time_points).
            max_len (int): The maximum number of latent frames to generate.

        Returns:
            A tuple containing:
            - torch.Tensor: Text/class prediction. Shape: (batch, num_classes).
            - torch.Tensor: Generated latent sequence. Shape: (batch, max_len, 4, 36, 64).
        """
        self.eval()
        device = src_eeg.device
        
        # 1. Process source (EEG) sequence
        b, seq_len_eeg, c, t = src_eeg.shape
        src_embedded = self.eeg_embedding(src_eeg.reshape(b * seq_len_eeg, 1, c, t))
        src_embedded = src_embedded.view(b, seq_len_eeg, self.d_model)
        src = self.positional_encoding(src_embedded)
        
        # 2. Run encoder
        encoder_output = self.transformer_encoder(src)
        
        # 3. Text prediction from encoder output
        txt_prediction = self.txt_predictor(torch.mean(encoder_output, dim=1))

        # 4. Auto-regressive decoding
        # Start with a zero tensor as the "start-of-sequence" token
        decoder_input = torch.zeros((b, 1, self.d_model), device=device)
        
        generated_sequence = []

        for _ in range(max_len):
            # Generate mask for the current sequence length
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(decoder_input.size(1)).to(device)
            
            # Get model output for the current sequence
            decoder_output = self.transformer_decoder(decoder_input, encoder_output, tgt_mask=tgt_mask)
            
            # Predict the next latent frame from the last time step
            next_latent_flat = self.latent_predictor(decoder_output[:, -1:])
            
            # Append the predicted frame (in its raw, un-embedded form) to our results
            generated_sequence.append(next_latent_flat.view(b, 1, 4, 36, 64))

            # Embed the prediction and append it to the decoder input for the next step
            next_latent_embedded = self.latent_embedding(next_latent_flat)
            decoder_input = torch.cat([decoder_input, next_latent_embedded], dim=1)

        output_latents = torch.cat(generated_sequence, dim=1)
        return txt_prediction, output_latents

## Inference function

In [180]:
def run_inference(model: nn.Module, test_eeg: torch.Tensor, device: torch.device, 
                  max_len: int) -> np.ndarray:
    """
    Runs inference on the test set and returns the generated latent sequence.
    
    Returns:
        np.ndarray: The predicted latent sequences.
    """
    model.eval()
    with torch.no_grad():
        _, latent_out = model.generate(test_eeg, max_len=max_len)
    
    return latent_out.cpu().numpy()

## Train function

In [181]:
def train_one_epoch(model: nn.Module, dataloader: DataLoader, loss_fn: nn.Module,
                    optimizer: torch.optim.Optimizer, scheduler: Any, device: torch.device) -> float:
    """
    对数据集执行一次完整的训练过程。

    Returns:
        float: 该轮次的平均损失。
    """
    model.train()
    total_loss = 0.0
    for eeg_seq, video_seq in tqdm(dataloader, desc="Training"):
        eeg_seq = eeg_seq.float().to(device)
        video_seq = video_seq.float().to(device)

        # Prepare target sequences for teacher forcing
        # Input to the decoder is the sequence shifted right, with a start token
        start_token = torch.zeros_like(video_seq[:, :1, ...]) # (b, 1, c, h, w)
        target_video_for_input = torch.cat([start_token, video_seq[:, :-1, ...]], dim=1)
        
        # Ground truth for loss calculation is the original sequence
        ground_truth_video = video_seq

        optimizer.zero_grad()
        
        _, pred_video_seq = model(eeg_seq, target_video_for_input)
        
        loss = loss_fn(pred_video_seq, ground_truth_video)
        loss.backward()
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

## Complete training function with model saving

In [182]:
def train():
    """
    完整的训练函数，包括数据加载、模型训练和最佳模型保存。
    
    该函数会：
    1. 加载和预处理训练数据
    2. 初始化模型、优化器和调度器
    3. 执行训练循环，跟踪最佳模型
    4. 保存最佳模型权重
    """
    # 创建输出目录
    output_dir = Path(CONFIG["save_path"]).parent
    output_dir.mkdir(parents=True, exist_ok=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # --- 数据准备 ---
    train_loader, _, _ = load_and_preprocess_data()

    # --- 模型初始化 ---
    model = LatentPredictionTransformer(
        d_model=CONFIG["d_model"],
        n_head=CONFIG["n_head"],
        encoder_layers=CONFIG["encoder_layers"],
        decoder_layers=CONFIG["decoder_layers"],
        dropout=CONFIG["dropout"],
        eeg_channels=CONFIG["eeg_channels"],
        eeg_t_window=CONFIG["eeg_t_window"]
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CONFIG["epochs"] * len(train_loader)
    )
    loss_fn = nn.MSELoss()

    # --- 训练循环 ---
    print("开始训练...")
    best_loss = float('inf')
    
    for epoch in range(CONFIG["epochs"]):
        epoch_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, scheduler, device)
        print(f"Epoch {epoch+1}/{CONFIG['epochs']}, Loss: {epoch_loss:.6f}")
        
        # 保存最佳模型
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_model_path = CONFIG["save_path"]
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': best_loss,
                'config': CONFIG
            }, best_model_path)
            print(f"保存最佳模型到 {best_model_path}，损失: {best_loss:.6f}")
    
    print(f"训练完成！最佳损失: {best_loss:.6f}")
    return model



## Complete test function with loss calculation

In [183]:
def test():
    """
    完整的测试函数，加载最佳模型并在测试集上进行评估。
    
    该函数会：
    1. 加载预处理的测试数据
    2. 加载训练好的最佳模型
    3. 在测试集上计算损失
    4. 生成预测结果并保存
    
    Returns:
        tuple: (test_loss, predicted_latents)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # --- 数据准备 ---
    _, test_eeg, test_latent_data = load_and_preprocess_data()
    
    # --- 模型初始化和加载 ---
    model = LatentPredictionTransformer(
        d_model=CONFIG["d_model"],
        n_head=CONFIG["n_head"],
        encoder_layers=CONFIG["encoder_layers"],
        decoder_layers=CONFIG["decoder_layers"],
        dropout=CONFIG["dropout"],
        eeg_channels=CONFIG["eeg_channels"],
        eeg_t_window=CONFIG["eeg_t_window"]
    ).to(device)
    
    # 加载最佳模型权重
    model_path = CONFIG["save_path"]
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"找不到训练好的模型文件: {model_path}")
    
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"成功加载模型权重，训练轮次: {checkpoint['epoch']}, 训练损失: {checkpoint['loss']:.6f}")
    
    # --- 测试集损失计算 ---
    model.eval()
    loss_fn = nn.MSELoss()
    
    with torch.no_grad():
        test_eeg_tensor = torch.from_numpy(test_eeg).float().to(device)
        test_latent_tensor = torch.from_numpy(test_latent_data).float().to(device)
        
        # 准备目标序列用于teacher forcing（仅用于损失计算）
        start_token = torch.zeros_like(test_latent_tensor[:, :1, ...])
        target_video_for_input = torch.cat([start_token, test_latent_tensor[:, :-1, ...]], dim=1)
        
        # 使用teacher forcing计算测试损失
        _, pred_video_seq = model(test_eeg_tensor, target_video_for_input)
        test_loss = loss_fn(pred_video_seq, test_latent_tensor).item()
        
        print(f"测试集损失 (Teacher Forcing): {test_loss:.6f}")
    
    # --- 自回归推理 ---
    print("开始自回归推理...")
    predicted_latents = run_inference(model, test_eeg_tensor, device, max_len=CONFIG["max_latent_seq_len"])
    
    print(f"预测潜在向量的形状: {predicted_latents.shape}")
    
    # --- 保存预测结果 ---
    output_path = CONFIG["result_path"]
    np.save(output_path, predicted_latents)
    print(f"已保存预测的潜在向量到 {output_path}")
    
    # 保存测试结果摘要
    test_results = {
        'test_loss': test_loss,
        'predicted_shape': predicted_latents.shape,
        'model_path': model_path,
        'config': CONFIG
    }
    
    for key, value in test_results.items():
        print(f"{key}: {value}")
    
    return test_loss, predicted_latents

## Execute Training

In [184]:
# 执行训练
if __name__ == "__main__":
    print("=== 开始训练阶段 ===")
    trained_model = train()
    print("\n=== 训练阶段完成 ===")

=== 开始训练阶段 ===
Using device: cpu
Train EEG shape: (1320, 7, 62, 100)
Train Latent shape: (1320, 12, 4, 36, 64)
Test EEG shape: (180, 7, 62, 100)
Test Latent shape: (180, 12, 4, 36, 64)
开始训练...


Training:  18%|█▊        | 2/11 [00:03<00:17,  1.98s/it]


KeyboardInterrupt: 

## Execute Testing

In [185]:
# 执行测试（需要先完成训练）
if __name__ == "__main__":
    print("\n=== 开始测试阶段 ===")
    test_loss, predictions = test()
    print(f"\n=== 测试阶段完成，最终测试损失: {test_loss:.6f} ===")


=== 开始测试阶段 ===
Using device: cpu
Train EEG shape: (1320, 7, 62, 100)
Train Latent shape: (1320, 12, 4, 36, 64)
Test EEG shape: (180, 7, 62, 100)
Test Latent shape: (180, 12, 4, 36, 64)
成功加载模型权重，训练轮次: 16, 训练损失: 0.684018
测试集损失 (Teacher Forcing): 0.938295
开始自回归推理...
预测潜在向量的形状: (180, 6, 4, 36, 64)
已保存预测的潜在向量到 data/metadata/seq2seq_prediction.pt
test_loss: 0.9382950067520142
predicted_shape: (180, 6, 4, 36, 64)
model_path: checkpoints/seq2seq/best_seq2seq_model.pt
config: {'EEG_data_path': 'data/sliced_eeg/watching', 'latent_data_path': 'data/metadata/videos_latents.pt', 'save_path': 'checkpoints/seq2seq/best_seq2seq_model.pt', 'result_path': 'data/metadata/seq2seq_prediction.pt', 'train_test': 0.88, 'video_latent_shape': (250, 12, 4, 36, 64), 'eeg_t_window': 100, 'eeg_t_step': 50, 'd_model': 512, 'eeg_channels': 62, 'n_head': 4, 'encoder_layers': 2, 'decoder_layers': 4, 'seed': 42, 'dropout': 0.1, 'batch_size': 128, 'learning_rate': 0.001, 'epochs': 100, 'max_latent_seq_len': 6}

=== 测试阶段